# Day 1 Sprint — SkillGraph Adaptive LLM

Run order:
1. Setup (GPU + deps)
2. Optional tiny three-model HF run (needs `HF_TOKEN`)
3. TRL GRPO training (GPU)
4. Display plots

In [ ]:
!nvidia-smi || echo "No GPU detected — switch Runtime to T4 GPU"

In [ ]:
REPO_URL = "https://github.com/Diyakalra1/skillgraph-adaptive-llm.git"
BRANCH = "main"

In [ ]:
!rm -rf repo
!git clone -b {BRANCH} {REPO_URL} repo
%cd repo
!pip -q install -r skillgraph_adaptive_env/training/requirements-trl.txt
!pip -q install -e skillgraph_adaptive_env

## Optional: tiny three-model HF run
Set `HF_TOKEN` in Colab secrets or env. Keep episodes small to save credits.

In [ ]:
import os
assert os.getenv("HF_TOKEN", "").strip(), "Set HF_TOKEN in Colab secrets first"

!python -m skillgraph_adaptive_env.training.run_training_three_models \
  --episodes 3 --seed 7 \
  --hf-token $HF_TOKEN \
  --max-tokens 64 --max-api-calls 40 \
  --out-dir training/runs/hf_three_models_day1

## TRL GRPO (main RL proof run)

In [ ]:
!python -m skillgraph_adaptive_env.training.run_training_trl_grpo \
  --episodes 30 \
  --seed 7 \
  --max-turns 12 \
  --model-id Qwen/Qwen2.5-0.5B-Instruct \
  --max-samples 90 \
  --epochs 1 \
  --out-dir training/runs/trl_grpo_day1

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

out = Path("training/runs/trl_grpo_day1")
print(json.dumps(json.loads((out / "summary.json").read_text()), indent=2))
print(json.dumps(json.loads((out / "eval_summary.json").read_text()), indent=2))

for name in ["reward_vs_steps.png", "success_rate_trend.png", "reward_components.png", "training_loss.png"]:
    p = out / "plots" / name
    if p.exists():
        print(name)
        display(Image(filename=str(p)))
    else:
        print("Missing:", name)